In [2]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import hashlib
import pandas as pd
import openpyxl


In [3]:
folder_path = r"C:\Users\KS\Desktop\ตรวจไฟล์"
data = []
bad_files = []

In [4]:
def clean_line(value):
    if pd.isna(value): return None
    
    val_str = "".join(str(value).split())
    parts = val_str.split(",")
    if len(parts) >= 5:
        return ",".join(parts[2:-1])
    return None


In [ ]:

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".xlsx") and not filename.startswith("~$"):
        filepath = os.path.join(folder_path, filename)
        
        try:
            # 1. ลองอ่านไฟล์แบบไม่ระบุคอลัมน์ก่อนเพื่อเช็คโครงสร้าง
            df_check = pd.read_excel(filepath, engine='openpyxl')
            
            # เช็คไฟล์ว่าง
            if df_check.empty:
                bad_files.append({"filename": filename, "reason": "ไฟล์ว่างเปล่า"})
                continue
            
            # เช็คว่ามีคอลัมน์ index 0 จริงไหม (ป้องกัน out-of-bounds)
            if len(df_check.columns) < 1:
                bad_files.append({"filename": filename, "reason": "ไม่มีคอลัมน์ข้อมูล"})
                continue

            # 2. เริ่มประมวลผลข้อมูล (เอาเฉพาะคอลัมน์แรก)
            col = df_check.iloc[:, 0].dropna().astype(str)
            col_cleaned = col.apply(clean_line).dropna()

            # เช็คว่าหลังจาก clean แล้วเหลือข้อมูลไหม (เช่น มีข้อมูลแต่คอมมาไม่ครบ 5 ตัวเลยสักบรรทัด)
            if col_cleaned.empty:
                bad_files.append({"filename": filename, "reason": "รูปแบบข้อมูล (Comma) ไม่ถูกต้อง"})
                continue

            # 3. ทำ Hash ตามปกติ
            col_sorted = col_cleaned.sort_values()
            combined = "\n".join(col_sorted)
            file_hash = hashlib.md5(combined.encode("utf-8")).hexdigest()

            data.append({
                "filename": filename,
                "hash": file_hash,
                "row_count": len(col_cleaned) # เก็บจำนวนบรรทัดไว้ดูเปรียบเทียบด้วย
            })
            
        except Exception as e:
            bad_files.append({"filename": filename, "reason": f"Error: {str(e)}"})

# --- ส่วนสรุปผล ---
print("=== สรุปไฟล์ที่ใช้งานได้ ===")
df_result = pd.DataFrame(data)
print(df_result)

if bad_files:
    print("\n⚠️ === พบไฟล์ที่ผิดปกติ (โปรดตรวจสอบ) ===")
    df_bad = pd.DataFrame(bad_files)
    print(df_bad)

=== สรุปไฟล์ที่ใช้งานได้ ===
                                              filename  \
0    ----------------------------------------------...   
1                                DS256903-00482-1.XLSX   
2                                  DS256903-00882.xlsx   
3                                  DS256903-01131.xlsx   
4                               K1-DS256903-00081.XLSX   
..                                                 ...   
177                             SP-DS256903-02551.XLSX   
178                           SP-DS256903-02646-1.XLSX   
179                             SP-DS256903-02646.XLSX   
180                             SP-DS256903-02743.XLSX   
181                       SP-DS6903-02665-1ของกอง.XLSX   

                                 hash  row_count  
0    269595623c11a05c551afdbfa076ec21        102  
1    69f71732b88b5cda55cb2c9651876361         41  
2    269595623c11a05c551afdbfa076ec21        102  
3    2651df0aebad74c6223f5678a9408c7b         83  
4    c0663b48d2d948

In [6]:
# หาไฟล์ที่ hash ซ้ำ
duplicate_files = df_result[df_result.duplicated("hash", keep=False)]


In [7]:
# จัดกลุ่มซ้ำ
grouped = (
    duplicate_files
    .groupby("hash")["filename"]
    .apply(list)
    .reset_index(drop=True)
)

grouped.apply(pd.Series)

,0,1,2
0,DS256903-01131.xlsx,SP-DS256903-01131.XLSX,NaN
1,----------------------------------------------...,DS256903-00882.xlsx,K3-DS256903-00882.XLSX
2,K3-DS256903-02572.XLSX,K4-DS256903-02572.XLSX,NaN
3,K3-DS256903-01173.XLSX,K5-DS256903-01173.XLSX,NaN
